# Phase 5B — Supervised Learning: Classification

**Algorithms covered:** Logistic Regression, KNN, Decision Trees, Random Forests, SVM, Naive Bayes.

**Dataset used:** Titanic (from seaborn) — predict survival.

**Install:** `pip install scikit-learn`

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

np.random.seed(42)
sns.set_theme(style="whitegrid")

In [ ]:
# Prepare Titanic dataset
df = sns.load_dataset("titanic")

# Feature selection and cleaning
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

df_model = df[features + [target]].copy()
df_model["age"].fillna(df_model["age"].median(), inplace=True)
df_model["fare"].fillna(df_model["fare"].median(), inplace=True)
df_model["embarked"].fillna(df_model["embarked"].mode()[0], inplace=True)

# Encode categoricals
df_model["sex"] = (df_model["sex"] == "female").astype(int)
df_model["embarked"] = LabelEncoder().fit_transform(df_model["embarked"])

X = df_model[features].values
y = df_model[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

---
## 1. Train and Compare All Classifiers

In [ ]:
# Build pipelines (scaler + model) so scaling is done correctly
models = {
    "Logistic Regression": Pipeline(
        [
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
    "KNN (k=5)": Pipeline(
        [("scl", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=5))]
    ),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=5, random_state=42
    ),
    "SVM": Pipeline(
        [("scl", StandardScaler()), ("clf", SVC(probability=True, random_state=42))]
    ),
    "Naive Bayes": GaussianNB(),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }

results_df = pd.DataFrame(results).T.round(4)
print(results_df.sort_values("f1", ascending=False))

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(11, 5))
results_df.plot(
    kind="bar", ax=ax, rot=30, color=["steelblue", "coral", "green", "gold"]
)
ax.set_ylim(0, 1.1)
ax.set_title("Classifier Comparison — Titanic")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

---
## 2. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        continue
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC=0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Classifiers")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Feature Importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=features).sort_values(
    ascending=True
)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature Importance — Random Forest")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

---
## 4. Detailed Look: Decision Tree

Decision Trees are fully interpretable — you can see exactly what decisions the model makes.

In [ ]:
dt = models["Decision Tree"]
tree_rules = export_text(dt, feature_names=features, max_depth=3)
print("Decision Tree Rules (first 3 levels):")
print(tree_rules)

---
## Algorithm Quick Reference

| Algorithm | Pros | Cons | When to Use |
|-----------|------|------|-------------|
| Logistic Regression | Fast, interpretable, probabilistic | Assumes linearity | Baseline, when interpretability matters |
| KNN | Simple, no training | Slow at predict time, sensitive to scale | Small datasets, non-linear boundaries |
| Decision Tree | Interpretable, handles mixed types | Overfits easily | Explaining decisions to stakeholders |
| Random Forest | Robust, good defaults, feature importance | Slower, less interpretable | General purpose, tabular data |
| SVM | Effective in high dimensions | Slow on large data, needs scaling | Text classification, small-medium datasets |
| Naive Bayes | Very fast, works with small data | Assumes feature independence | Text, spam detection |